In [2]:
import os
from dotenv import load_dotenv
import json
import pathlib
import textwrap
import google.generativeai as genai
from IPython.display import display
from IPython.display import Markdown
from google.api_core import retry
import typing_extensions as typing


def to_markdown(text):
    text = text.replace('•', '  *')
    return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

load_dotenv()
google_api_key = os.getenv('GEMINI_API_KEY')

genai.configure(api_key=google_api_key)

system_prompt ={"text":"""You are a medical AI Agent. Tell is there any food which can react with the drugs in the list in 100 words"""}


# tool to add medicine names to a list
medicine_list = []
def medicine_tool(medicine):
    med = medicine.lower().strip()
    if med not in medicine_list:
        medicine_list.append(med)
    return medicine_list
    
medicine_class={
    "name": "medicine_tool",
    "description":"gets the list of medicines",
    "parameters":{
        "type":"object",
        "properties":{
            "medicine":{
                "type":"string",
                "description":"The medicine name",
            },
        },
        "required":["medicine"],
        "additionalProperties":False
    }
}

tool_config = {
    "function_calling_config": {
        "mode": "ANY",
        "allowed_function_names": ["medicine_class"]
    }
}

user_prompt=f"reaction of food and drink with drugs in the list:\n"
user_prompt+=str([medicine_class])

def message_gemini(*user_prompt):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    gemini = genai.GenerativeModel(
    model_name='gemini-1.5-flash',
    system_instruction=system_prompt,
    tool_config=tool_config
)
    response = gemini.generate_content(user_prompt)
    return response.text


In [3]:
to_markdown(message_gemini("paracetamol","flexon",))

> Paracetamol (acetaminophen) and Flexon (containing ibuprofen) are commonly combined in over-the-counter pain relievers.  No specific foods are known to directly interact with either drug in a dangerous way. However,  alcohol consumption while taking these medications can increase the risk of liver damage with paracetamol and stomach upset with ibuprofen.  Large quantities of caffeine, often found in coffee and tea, might exacerbate potential side effects like anxiety or insomnia associated with these drugs.  Always consult a doctor or pharmacist for specific advice, particularly if you have pre-existing health conditions.
